# Player Performance Prediction — EDA on Cleaned Data
## Group 1 — Team Project (Week 4)

**Objective:** Analyze pre-cleaned IPL and BCCI datasets with professional visualizations.  
Players identified by unique `player_id` — no player names.

### Pipeline Overview
| Stage | Content |
|-------|---------|
| **1** | Load Cleaned IPL Data (features) |
| **2** | Clean & Load BCCI Data |
| **3** | Univariate Analysis — distributions |
| **4** | Bivariate & Multivariate — correlations, heatmaps |
| **5** | IPL Season Trends |
| **6** | BCCI Format Analysis |
| **7** | Engineered Feature Analysis |
| **8** | Target Variable Deep Dive |
| **9** | Train-Test Split |
| **10**| Key Findings

In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
import os, re, warnings
warnings.filterwarnings('ignore')
sns.set_theme(style='whitegrid', palette='Set2')
plt.rcParams['figure.dpi'] = 120
plt.rcParams['savefig.dpi'] = 150
plt.rcParams['font.size'] = 11
%matplotlib inline
print('Setup complete')

---
## 1. Load Cleaned IPL Data

In [ ]:
DATA_DIR = os.path.dirname(os.path.abspath('.'))
feat_path = os.path.join(DATA_DIR, '03_eda_cleaned', 'ipl_features.csv')
clean_path = os.path.join(DATA_DIR, '03_eda_cleaned', 'ipl_cleaned.csv')
raw_dir = os.path.join(DATA_DIR, '01_raw_data_and_scrapers')

# Try alternative paths
for p in [feat_path, 'ipl_features.csv', '../03_eda_cleaned/ipl_features.csv']:
    if os.path.exists(p):
        feat_path = p
        break
for p in [clean_path, 'ipl_cleaned.csv', '../03_eda_cleaned/ipl_cleaned.csv']:
    if os.path.exists(p):
        clean_path = p
        break

df_feat = pd.read_csv(feat_path)
df_clean = pd.read_csv(clean_path)

print('=' * 60)
print('IPL FEATURE-ENGINEERED DATASET')
print('=' * 60)
print(f'Shape: {df_feat.shape[0]:,} rows x {df_feat.shape[1]} columns')
print(f'Unique player_ids: {df_feat["player_id"].nunique():,}')
print(f'Seasons: {int(df_feat["season"].min())} - {int(df_feat["season"].max())}')
print(f'player_name in data: {"player_name" in df_feat.columns}')
print(f'_outlier flagged: {df_feat["_outlier"].sum()}')
print()
print('Columns:')
for c in df_feat.columns:
    print(f'  {c:35s} {str(df_feat[c].dtype):10s}')

---
## 2. Clean & Load BCCI Data

In [ ]:
bcci_raw_paths = [
    os.path.join(raw_dir, 'bcci_stats_rankings_all.csv'),
    '../01_raw_data_and_scrapers/bcci_stats_rankings_all.csv',
    os.path.join(DATA_DIR, '01_raw_data_and_scrapers', 'bcci_stats_rankings_all.csv'),
]
bcci_path = None
for p in bcci_raw_paths:
    if os.path.exists(p):
        bcci_path = p
        break

if bcci_path:
    df_bcci_raw = pd.read_csv(bcci_path)
    print(f'BCCI raw data loaded: {df_bcci_raw.shape}')
else:
    print('BCCI raw data not found. Place bcci_stats_rankings_all.csv in 01_raw_data_and_scrapers/')
    df_bcci_raw = None

In [ ]:
def clean_bcci(df):
    if df is None:
        return None, {}
    df = df.copy()
    df.columns = [c.replace('\u2019','').replace('\u2018','').replace("'",'').replace("`",'').replace(" ",'_').lower() for c in df.columns]
    batting_cols = ['batting_matches','batting_inns','batting_runs','batting_avg',
                    'batting_sr','batting_hs','batting_4s','batting_6s','batting_50s','batting_100s']
    bowling_cols = ['bowling_matches','bowling_inns','bowling_wickets','bowling_avg',
                    'bowling_economy_rate','bowling_sr','bowling_runs']
    all_stats = batting_cols + bowling_cols
    keep = ['format','player_name'] + [c for c in all_stats if c in df.columns]
    df = df[keep]
    for c in all_stats:
        if c in df.columns:
            df[c] = pd.to_numeric(df[c], errors='coerce')
    agg = {c: 'max' for c in all_stats if c in df.columns}
    pivot = df.groupby(['player_name','format']).agg(agg).reset_index().fillna(0)
    players = sorted(pivot['player_name'].unique())
    mapping = {n: i+1 for i, n in enumerate(players)}
    pivot['player_id'] = pivot['player_name'].map(mapping)
    pivot = pivot.drop(columns=['player_name'])
    return pivot, mapping

df_bcci, bcci_map = clean_bcci(df_bcci_raw)
if df_bcci is not None:
    print(f'BCCI cleaned: {df_bcci.shape[0]:,} rows, {df_bcci["player_id"].nunique():,} unique players')
    print(f'Formats: {list(df_bcci["format"].unique())}')
    print(f'player_name dropped: True')
    print(f'Columns: {list(df_bcci.columns)}')

---
## 3. Univariate Analysis — IPL

In [ ]:
fig, axes = plt.subplots(2, 4, figsize=(22, 11))
fig.suptitle('IPL — Batting Metrics Distribution (Cleaned Data)', fontsize=16, fontweight='bold', y=1.02)

configs = [
    ('runs', 'Runs', '#2E86C1'), ('batting_average', 'Batting Average', '#28B463'),
    ('strike_rate', 'Strike Rate', '#D4AC0D'), ('fours', 'Fours', '#CB4335'),
    ('sixes', 'Sixes', '#884EA0'), ('fifties', 'Fifties', '#1ABC9C'),
    ('hundreds', 'Hundreds', '#E67E22'), ('matches', 'Matches Played', '#5D6D7E'),
]
for ax, (col, title, color) in zip(axes.flat, configs):
    data = df_feat[col].dropna()
    sns.histplot(data, bins=40, kde=True, color=color, edgecolor='white', linewidth=0.5, ax=ax)
    ax.axvline(data.median(), color='red', ls='--', lw=1.5, label=f'Median: {data.median():.1f}')
    ax.axvline(data.mean(), color='darkblue', ls=':', lw=1.5, label=f'Mean: {data.mean():.1f}')
    ax.set_title(title, fontsize=12, fontweight='bold')
    ax.set_ylabel('Frequency')
    ax.legend(fontsize=7)
plt.tight_layout()
plt.show()

In [ ]:
fig, axes = plt.subplots(1, 4, figsize=(20, 5.5))
fig.suptitle('IPL — Bowling Metrics Distribution (Cleaned Data)', fontsize=16, fontweight='bold', y=1.08)

configs = [
    ('wickets', 'Wickets', '#2E86C1'), ('bowling_average', 'Bowling Average', '#28B463'),
    ('economy_rate', 'Economy Rate', '#D4AC0D'), ('bowling_strike_rate', 'Bowling SR', '#CB4335'),
]
for ax, (col, title, color) in zip(axes.flat, configs):
    data = df_feat[col].dropna()
    sns.histplot(data, bins=40, kde=True, color=color, edgecolor='white', linewidth=0.5, ax=ax)
    ax.axvline(data.median(), color='red', ls='--', lw=1.5, label=f'Median: {data.median():.1f}')
    ax.axvline(data.mean(), color='darkblue', ls=':', lw=1.5, label=f'Mean: {data.mean():.1f}')
    ax.set_title(title, fontsize=12, fontweight='bold')
    ax.set_ylabel('Frequency')
    ax.legend(fontsize=7)
plt.tight_layout()
plt.show()

In [ ]:
fig, axes = plt.subplots(2, 4, figsize=(22, 10))
fig.suptitle('IPL — Box Plots (Outlier Visibility)', fontsize=16, fontweight='bold', y=1.02)
cols = [('runs','#2E86C1'),('batting_average','#28B463'),('strike_rate','#D4AC0D'),
        ('fours','#CB4335'),('wickets','#884EA0'),('economy_rate','#1ABC9C'),
        ('bowling_strike_rate','#E67E22'),('matches','#5D6D7E')]
for ax, (col, color) in zip(axes.flat, cols):
    data = df_feat[col].dropna()
    ax.boxplot(data, patch_artist=True, widths=0.4,
               boxprops=dict(facecolor=color, alpha=0.6),
               medianprops=dict(color='red', lw=2),
               flierprops=dict(marker='o', markerfacecolor='red', markersize=4, alpha=0.5))
    ax.set_title(col.replace('_',' ').title(), fontsize=11, fontweight='bold')
    ax.set_xticks([])
    q1, q3 = np.percentile(data, 25), np.percentile(data, 75)
    ax.text(0.95, 0.95, f'N={len(data):,}\nIQR={q3-q1:.1f}', transform=ax.transAxes, fontsize=7,
            va='top', ha='right', bbox=dict(boxstyle='round', facecolor='wheat', alpha=0.5))
plt.tight_layout()
plt.show()

---
## 4. Bivariate & Multivariate Analysis

In [ ]:
num_cols = ['matches','innings','runs','batting_average','strike_rate',
            'fours','sixes','fifties','hundreds','catches',
            'wickets','bowling_average','economy_rate','bowling_strike_rate']
corr = df_feat[num_cols].corr()
fig, ax = plt.subplots(figsize=(16, 12))
mask = np.triu(np.ones_like(corr, dtype=bool), k=1)
sns.heatmap(corr, mask=mask, annot=True, fmt='.2f', cmap=sns.diverging_palette(230, 20, as_cmap=True),
            center=0, square=True, linewidths=0.8, cbar_kws={'shrink': 0.75, 'label': 'Pearson r'},
            vmin=-1, vmax=1, ax=ax)
ax.set_title('IPL — Feature Correlation Matrix (Cleaned Data)', fontsize=16, fontweight='bold', pad=20)
ax.set_xticklabels(ax.get_xticklabels(), rotation=45, ha='right', fontsize=9)
ax.set_yticklabels(ax.get_yticklabels(), rotation=0, fontsize=9)
plt.tight_layout()
plt.show()

In [ ]:
corr_vals = corr.unstack().reset_index()
corr_vals.columns = ['F1','F2','r']
corr_vals = corr_vals[corr_vals['F1'] != corr_vals['F2']]
corr_vals['Abs'] = corr_vals['r'].abs()
corr_vals = corr_vals.drop_duplicates(subset=['Abs']).sort_values('Abs', ascending=False)

fig, axes = plt.subplots(1, 2, figsize=(16, 6))
fig.suptitle('IPL — Strongest Feature Correlations', fontsize=16, fontweight='bold', y=1.05)

top_pos = corr_vals[corr_vals['r'] > 0].head(8)
axes[0].barh(range(len(top_pos)), top_pos['r'].values, color='#27AE60', edgecolor='white')
axes[0].set_yticks(range(len(top_pos)))
axes[0].set_yticklabels([f'{a} vs {b}' for a,b in zip(top_pos['F1'],top_pos['F2'])], fontsize=9)
axes[0].set_title('Top Positive Correlations', fontsize=13, fontweight='bold')
axes[0].set_xlabel('Pearson r')
axes[0].axvline(0, color='black', lw=0.5)

top_neg = corr_vals[corr_vals['r'] < 0].tail(8)
axes[1].barh(range(len(top_neg)), top_neg['r'].values, color='#E74C3C', edgecolor='white')
axes[1].set_yticks(range(len(top_neg)))
axes[1].set_yticklabels([f'{a} vs {b}' for a,b in zip(top_neg['F1'],top_neg['F2'])], fontsize=9)
axes[1].set_title('Top Negative Correlations', fontsize=13, fontweight='bold')
axes[1].set_xlabel('Pearson r')
axes[1].axvline(0, color='black', lw=0.5)
plt.tight_layout()
plt.show()

In [ ]:
key_cols = ['runs','batting_average','strike_rate','wickets','economy_rate','matches']
sample = df_feat[key_cols].sample(min(500, len(df_feat)), random_state=42)
g = sns.pairplot(sample, diag_kind='kde',
                 plot_kws={'alpha':0.4, 's':15, 'color':'#2E86C1'},
                 diag_kws={'color':'#2E86C1', 'fill':True})
g.fig.suptitle('IPL — Pairwise Relationships (500-row sample)', fontsize=14, fontweight='bold', y=1.02)
plt.show()

---
## 5. IPL Season Trends

In [ ]:
season_stats = df_feat.groupby('season').agg(
    players=('player_id', 'nunique'), avg_runs=('runs', 'mean'),
    total_runs=('runs', 'sum'), avg_sr=('strike_rate', 'mean'),
    avg_ba=('batting_average', 'mean'), total_wickets=('wickets', 'sum'),
    avg_econ=('economy_rate', 'mean'), total_matches=('matches', 'sum'),
).reset_index()

fig, axes = plt.subplots(2, 4, figsize=(22, 12))
fig.suptitle('IPL Season-wise Trends (2008–2026)', fontsize=18, fontweight='bold', y=1.02)
trends = [('players','Unique Players','#2E86C1'),('avg_runs','Avg Runs/Player','#28B463'),
          ('total_runs','Total Runs','#D4AC0D'),('avg_sr','Avg Strike Rate','#CB4335'),
          ('avg_ba','Avg Batting Avg','#884EA0'),('total_wickets','Total Wickets','#1ABC9C'),
          ('avg_econ','Avg Economy Rate','#E67E22'),('total_matches','Total Matches','#5D6D7E')]
for ax, (col, title, color) in zip(axes.flat, trends):
    ax.plot(season_stats['season'], season_stats[col], marker='o', color=color,
            linewidth=2, markersize=6, markerfacecolor='white', markeredgecolor=color, markeredgewidth=2)
    ax.set_title(title, fontsize=12, fontweight='bold')
    ax.set_xlabel('Season')
    ax.tick_params(axis='x', rotation=45)
    ax.grid(True, alpha=0.3)
plt.tight_layout()
plt.show()

---
## 6. BCCI Format Analysis

In [ ]:
if df_bcci is not None:
    fig, axes = plt.subplots(1, 3, figsize=(18, 6))
    fig.suptitle('BCCI — Format-wise Stats Distribution', fontsize=16, fontweight='bold', y=1.05)
    for ax, fmt, color in zip(axes, ['TEST','ODI','T20I'], ['#3498DB','#2ECC71','#E74C3C']):
        sub = df_bcci[df_bcci['format'] == fmt]
        if 'batting_runs' in sub.columns:
            data = sub[sub['batting_runs'] > 0]['batting_runs']
            if len(data) > 0:
                sns.histplot(data, bins=30, kde=True, color=color, edgecolor='white', ax=ax)
                ax.axvline(data.median(), color='red', ls='--', label=f'Median: {data.median():,.0f}')
                ax.legend(fontsize=8)
        ax.set_title(f'{fmt} — Batting Runs (n={len(sub)})', fontsize=13, fontweight='bold')
        ax.set_xlabel('Runs')
    plt.tight_layout()
    plt.show()
else:
    print('BCCI data not loaded. Skipping format analysis.')

In [ ]:
if df_bcci is not None:
    fig, axes = plt.subplots(1, 3, figsize=(18, 5.5))
    fig.suptitle('BCCI — Bowling Wickets by Format', fontsize=16, fontweight='bold', y=1.05)
    for ax, fmt, color in zip(axes, ['TEST','ODI','T20I'], ['#3498DB','#2ECC71','#E74C3C']):
        sub = df_bcci[df_bcci['format'] == fmt]
        if 'bowling_wickets' in sub.columns:
            data = pd.to_numeric(sub['bowling_wickets'], errors='coerce').dropna()
            data = data[data > 0]
            if len(data) > 0:
                sns.histplot(data, bins=30, kde=True, color=color, edgecolor='white', ax=ax)
                ax.axvline(data.median(), color='red', ls='--', label=f'Median: {data.median():.0f}')
                ax.legend(fontsize=8)
        ax.set_title(f'{fmt} — Wickets (n={len(data) if len(data) > 0 else 0})', fontsize=13, fontweight='bold')
        ax.set_xlabel('Wickets')
    plt.tight_layout()
    plt.show()
else:
    print('BCCI data not loaded.')

---
## 7. Engineered Feature Analysis

In [ ]:
fig, axes = plt.subplots(2, 3, figsize=(20, 11))
fig.suptitle('Engineered Feature Distributions', fontsize=16, fontweight='bold', y=1.02)

eng = [('batting_impact','Batting Impact','#2E86C1'),
       ('bowling_impact','Bowling Impact','#28B463'),
       ('consistency_score','Consistency Score','#D4AC0D'),
       ('total_runs','Total Runs','#CB4335'),
       ('total_wickets','Total Wickets','#884EA0'),
       ('overall_performance_score','Target: Performance Score','#1ABC9C')]
for ax, (col, title, color) in zip(axes.flat, eng):
    data = df_feat[col]
    if col == 'bowling_impact':
        data = data[data > 0]
    sns.histplot(data, bins=50, kde=True, color=color, edgecolor='white', linewidth=0.5, ax=ax)
    ax.axvline(data.median(), color='red', ls='--', lw=1.5, label=f'Median: {data.median():.2f}')
    ax.axvline(data.mean(), color='darkblue', ls=':', lw=1.5, label=f'Mean: {data.mean():.2f}')
    ax.set_title(title, fontsize=12, fontweight='bold')
    ax.set_ylabel('Frequency')
    ax.legend(fontsize=7)
plt.tight_layout()
plt.show()

In [ ]:
cat_order = ['Poor','Average','Good','Very Good','Excellent']
cat_counts = df_feat['performance_category'].value_counts()
cat_counts = cat_counts.reindex([c for c in cat_order if c in cat_counts.index])
cat_colors = ['#E74C3C','#F39C12','#F1C40F','#2ECC71','#27AE60']

fig, axes = plt.subplots(1, 3, figsize=(18, 6))
fig.suptitle('Performance Category Analysis', fontsize=16, fontweight='bold', y=1.05)

axes[0].bar(cat_counts.index, cat_counts.values, color=cat_colors, edgecolor='white', linewidth=1.5)
axes[0].set_title('Category Frequency', fontsize=13, fontweight='bold')
axes[0].set_ylabel('Count')
for bar, val in zip(axes[0].patches, cat_counts.values):
    axes[0].text(bar.get_x()+bar.get_width()/2, bar.get_height()+5, str(val), ha='center', fontweight='bold')

axes[1].pie(cat_counts.values, labels=cat_counts.index, autopct='%1.1f%%',
            colors=cat_colors, startangle=90, explode=[0.02]*len(cat_counts))
axes[1].set_title('Category Proportion', fontsize=13, fontweight='bold')

bp_data = [df_feat[df_feat['performance_category']==c]['overall_performance_score'].values for c in cat_counts.index]
bp = axes[2].boxplot(bp_data, patch_artist=True, labels=cat_counts.index, widths=0.5)
for patch, color in zip(bp['boxes'], cat_colors):
    patch.set_facecolor(color)
    patch.set_alpha(0.6)
axes[2].set_title('Score by Category', fontsize=13, fontweight='bold')
axes[2].set_ylabel('Score')
axes[2].tick_params(axis='x', rotation=30)
plt.tight_layout()
plt.show()

---
## 8. Target Variable Deep Dive

In [ ]:
target_cols = ['batting_impact','bowling_impact','consistency_score',
               'total_runs','total_wickets','runs','wickets',
               'batting_average','strike_rate','economy_rate',
               'bowling_average','bowling_strike_rate',
               'overall_performance_score']
target_corr = df_feat[target_cols].corr()['overall_performance_score'].sort_values(ascending=False)
target_plot = target_corr.drop('overall_performance_score')

fig, ax = plt.subplots(figsize=(12, 6))
colors = ['#27AE60' if v >= 0 else '#E74C3C' for v in target_plot.values]
target_plot.plot(kind='bar', color=colors, ax=ax, edgecolor='white', width=0.7)
ax.set_title('Feature Correlation with Target (Overall Performance Score)', fontsize=14, fontweight='bold')
ax.set_xlabel('Features')
ax.set_ylabel('Pearson Correlation')
ax.axhline(y=0, color='black', linewidth=0.5)
ax.axhline(y=0.5, color='gray', linestyle=':', lw=0.8, alpha=0.5)
ax.axhline(y=-0.5, color='gray', linestyle=':', lw=0.8, alpha=0.5)
plt.xticks(rotation=45, ha='right', fontsize=10)
for i, v in enumerate(target_plot.values):
    ax.text(i, v + (0.02 if v >= 0 else -0.06), f'{v:.3f}', ha='center', fontsize=8, fontweight='bold')
plt.tight_layout()
plt.show()

In [ ]:
top_impact = df_feat.groupby('player_id')['batting_impact'].max().sort_values(ascending=False).head(12)
top_bowl = df_feat.groupby('player_id')['bowling_impact'].max().sort_values(ascending=False).head(12)
top_cons = df_feat.groupby('player_id')['consistency_score'].max().sort_values(ascending=False).head(12)

fig, axes = plt.subplots(1, 3, figsize=(18, 6))
fig.suptitle('Top Performers by Engineered Metrics (player_id)', fontsize=14, fontweight='bold', y=1.02)
for ax, data, title, color in zip(axes, [top_impact, top_bowl, top_cons],
    ['Batting Impact','Bowling Impact','Consistency Score'], ['#2E86C1','#E74C3C','#28B463']):
    ax.barh(range(len(data)), data.values, color=color, edgecolor='white')
    ax.set_yticks(range(len(data)))
    ax.set_yticklabels([f'ID {int(pid)}' for pid in data.index])
    ax.set_title(f'Top 12 — {title}', fontsize=12, fontweight='bold')
    ax.invert_yaxis()
    ax.set_xlabel('Score')
plt.tight_layout()
plt.show()

---
## 9. Train-Test Split Summary

In [ ]:
from sklearn.model_selection import train_test_split

exclude = ['player_id','season','highest_score','best_bowling','performance_category','_outlier',
           'total_runs','total_wickets','batting_impact','bowling_impact','consistency_score',
           'overall_performance_score','highest_score_num','highest_score_notout',
           'best_bowling_wickets','best_bowling_runs','runs','wickets','matches','innings',
           'career_matches','career_runs','career_wickets','career_innings','seasons_played']

feat_cols = [c for c in df_feat.columns if c not in exclude and df_feat[c].dtype in (np.int64, np.float64)]
X = df_feat[feat_cols].fillna(0)
y = df_feat['overall_performance_score'].values
X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, random_state=42)

print('=' * 50)
print('TRAIN-TEST SPLIT SUMMARY')
print('=' * 50)
print(f'Total samples:    {len(X):,}')
print(f'Training set:     {len(X_train):,} ({len(X_train)/len(X)*100:.0f}%)')
print(f'Test set:         {len(X_test):,} ({len(X_test)/len(X)*100:.0f}%)')
print(f'Features:         {len(feat_cols)}')
print(f'Target:           overall_performance_score [0-10]')
print(f'player_id used:   Yes (no player_name in data)')
print()
print(f'Training features ({len(feat_cols)}):')
for i, c in enumerate(feat_cols, 1):
    print(f'  {i:2d}. {c}')

---
## 10. Key Findings & Summary

In [ ]:
fig, ax = plt.subplots(figsize=(16, 9))
ax.axis('off')

findings = [
    ('📊 Dataset Overview', [
        f'IPL: {df_feat.shape[0]:,} rows, {df_feat["player_id"].nunique():,} unique players, {int(df_feat["season"].min())}–{int(df_feat["season"].max())}',
        f'BCCI: {df_bcci.shape[0] if df_bcci is not None else 0:,} rows, {len(bcci_map) if df_bcci is not None else 0} players across TEST/ODI/T20I',
        f'Data is pre-cleaned: player_name replaced with player_id, outliers flagged ({df_feat["_outlier"].sum()} rows)',
        f'Feature-engineered: {len(df_feat.columns)} columns including target variable',
    ]),
    ('📈 Distributions', [
        'Runs, wickets, batting metrics are right-skewed (Pareto principle — few star performers)',
        'Most IPL players score < 200 runs or take < 5 wickets per season',
        'Strike rates cluster 120–140; economy rates 7–9',
        'BCCI: TEST runs spread widest, T20I most compact',
    ]),
    ('🔗 Key Correlations', [
        f'Runs ↔ Batting Avg: {df_feat["runs"].corr(df_feat["batting_average"]):.2f}',
        f'Runs ↔ Fours: {df_feat["runs"].corr(df_feat["fours"]):.2f}',
        f'Wickets ↔ Bowl Avg: {df_feat["wickets"].corr(df_feat["bowling_average"]):.2f}',
        'Engineered features (batting_impact, bowling_impact) strongly correlate with target',
    ]),
    ('📅 IPL Season Trends', [
        'Player participation grew from ~150 (2008) to ~450+ (2015–2019)',
        'Average runs per player show upward trend (improved T20 batting)',
        'Economy rates increased marginally (batting-friendly evolution)',
    ]),
    ('⚙ Feature Engineering', [
        f'Target: overall_performance_score [0-10] — composite of batting_impact, bowling_impact, consistency_score',
        f'Training features ({len(feat_cols)}): raw batting/bowling stats + career milestones',
        f'Train-test split: {len(X_train)} train / {len(X_test)} test ({len(X_test)/len(X)*100:.0f}% held out)',
        'Ready for model training (Phase 2)',
    ]),
]
y_pos = 0.95
for title, items in findings:
    ax.text(0.02, y_pos, title, fontsize=13, fontweight='bold', va='top', color='#2C3E50')
    y_pos -= 0.04
    for item in items:
        ax.text(0.05, y_pos, f'• {item}', fontsize=10, va='top', color='#555')
        y_pos -= 0.03
    y_pos -= 0.015
ax.set_title('EDA on Cleaned Data — Key Findings', fontsize=18, fontweight='bold', pad=20)
plt.show()